# Session 12 — Bias-Variance Tradeoff

**Goal:** answer the question Session 11 left open — *why* does the held-out curve rise
and then fall? — by measuring the two competing error sources separately on this
registry, and then using the diagnosis to pick a fix that actually addresses the
dominant one.

## What this stage does for the system

Session 11 established *that* there is an optimal complexity and showed how to find it
empirically. This session explains the shape, which matters because the two error
sources have opposite cures:

- **Bias** — error from a model too rigid to represent the real pattern. It shows up as
  predictions that are consistently off in the same direction, and it does *not* shrink
  with more data. The cure is more flexibility.
- **Variance** — error from a model so flexible it fits the particular patients it saw.
  It shows up as wildly different predictions for the same patient depending on which
  training sample was used. The cure is less flexibility, more data, or averaging.

Apply the wrong cure and you make things worse: adding complexity to a
variance-dominated model, or collecting more data to fix a bias problem that more data
cannot touch. Step 6 identifies which one dominates here and Step 7 attacks it directly.

This is also Session 4's bias-and-variance distinction returning in a new place — there
it was about sampling designs, here about model complexity, and it is the same
decomposition of error both times.

## The dataset

Every session in this module works on one registry: the UCI **Heart Disease**
dataset (Cleveland), fetched live from the UCI ML Repository with `ucimlrepo` so the
notebooks are runnable by anyone without a CSV sitting on their machine. It holds 303
patients with clinical measurements (`age`, `trestbps` resting blood pressure, `chol`
serum cholesterol, `thalach` max heart rate achieved, `oldpeak` ST depression),
categorical findings (`sex`, `cp` chest-pain type, `fbs` fasting blood sugar > 120,
`restecg`, `exang` exercise-induced angina, `slope`, `ca`, `thal`), and the outcome
`num` — angiographic disease severity 0-4, which this module binarises into
`target` (0 = no disease, 1 = disease present).

Deliberately one dataset throughout: switching datasets between topics would mean
re-learning the data every session instead of building cumulative familiarity with
one problem, the way a real analyst does.

## How to read this notebook

Every code cell is followed by a short **Observe / Infer** note: *Observe* points at
exactly what to look at in that cell's output, and *Infer* explains what conclusion to
draw from it — and what a different result would imply. Read them before running the
next cell; several of them flag things worth double-checking before you move on.

## Prerequisites

This session runs entirely locally — no account or credentials needed.

```bash
pip install ucimlrepo pandas numpy scipy scikit-learn statsmodels matplotlib seaborn
```

## Step 1 — Load the registry from the UCI repository

Fetching directly from the UCI ML Repository keeps this notebook runnable by anyone,
instead of depending on a CSV already sitting on your machine. The same nine lines
open every session in this module, so the 297 patients below are the identical 297
patients every other notebook analyses.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

heart_disease = fetch_ucirepo(id=45)
df = pd.concat([heart_disease.data.features, heart_disease.data.targets], axis=1)

# `num` is severity 0-4; this module screens for disease presence, so binarise it.
df = df.dropna().reset_index(drop=True)
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns="num")

print(f"{len(df)} patients, {len(df.columns)} columns")
print(f"disease prevalence: {df['target'].mean():.3f}")
df.head()

**Observe:** `297 patients, 14 columns` and `disease prevalence: 0.461`. The preview
shows `age`, `sex`, `cp`, `trestbps`, `chol`, `fbs`, `restecg`, `thalach`, `exang`,
`oldpeak`, `slope`, `ca`, `thal`, and the `target` column just derived.
**Infer:** 303 rows are fetched and 297 survive `dropna()` — six patients are missing
`ca` (number of major vessels seen on fluoroscopy) or `thal`. Dropping six rows out of
303 is defensible here and keeps every notebook in this module working on the identical
297 patients; on a larger fraction of missing values you would have to impute instead,
and *that* choice would itself need the distribution work of Session 3. If your row
count is not 297, you are on a different subset than every number quoted below.

## Step 2 — Set up the measurement

Bias and variance are properties of a *training procedure*, not of one fitted model, so
measuring them requires many training sets. Bootstrap resampling supplies them: draw
with replacement from the training patients, fit, predict on the same fixed test set,
and repeat. The spread of predictions for one test patient across those fits is
variance; the distance of their average from the truth is bias.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

features = [c for c in df.columns if c != "target"]
X = df[features].to_numpy()
y = df["target"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

N_BOOTSTRAP = 50
rng = np.random.default_rng(0)

def bootstrap_predictions(max_depth, n=N_BOOTSTRAP):
    """Fit n trees on n resampled training sets; return their test-set predictions."""
    preds = []
    for _ in range(n):
        idx = rng.integers(0, len(X_train), len(X_train))
        tree = DecisionTreeClassifier(max_depth=max_depth, random_state=0)
        tree.fit(X_train[idx], y_train[idx])
        preds.append(tree.predict_proba(X_test)[:, 1])
    return np.array(preds)

shallow = bootstrap_predictions(max_depth=1)
deep = bootstrap_predictions(max_depth=None)
print(f"shallow (depth 1): {shallow.shape[0]} fits x {shallow.shape[1]} test patients")
print(f"deep (unlimited):  {deep.shape[0]} fits x {deep.shape[1]} test patients")

**Observe:** two prediction matrices, each 50 fits by 60 test patients.
**Infer:** every row is a model that a slightly different registry would have produced,
which is what makes the column-wise spread a measurement of variance rather than a
metaphor for it. One limitation to keep in view: bootstrap resamples are drawn from the
same 237 training patients, so they share more structure than 50 genuinely independent
registries would, and the variance measured below is therefore an *underestimate*.
Directionally it is right, and the comparison between depths — which is what the session
turns on — is unaffected.

## Step 3 — Variance: how much do predictions for one patient disagree?

Pick a single test patient and look at the 50 predicted probabilities they receive.

In [ ]:
import matplotlib.pyplot as plt

patient = 0
print(f"test patient #{patient} (actual outcome: {y_test[patient]})")
print(f"  shallow tree: mean {shallow[:, patient].mean():.3f}, "
      f"range [{shallow[:, patient].min():.3f}, {shallow[:, patient].max():.3f}], "
      f"sd {shallow[:, patient].std():.3f}")
print(f"  deep tree:    mean {deep[:, patient].mean():.3f}, "
      f"range [{deep[:, patient].min():.3f}, {deep[:, patient].max():.3f}], "
      f"sd {deep[:, patient].std():.3f}")

print(f"\naverage per-patient sd across all {len(y_test)} test patients:")
print(f"  shallow: {shallow.std(axis=0).mean():.4f}")
print(f"  deep:    {deep.std(axis=0).mean():.4f}")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(shallow[:, patient], bins=15, alpha=0.65, color="steelblue", label="depth 1")
ax.hist(deep[:, patient], bins=15, alpha=0.65, color="indianred", label="unlimited depth")
ax.set_xlabel(f"predicted P(disease) for test patient #{patient}")
ax.set_ylabel("number of bootstrap fits")
ax.set_title("Same patient, 50 training sets, two model complexities")
ax.legend()
plt.tight_layout()
plt.show()

**Observe:** the shallow tree's 50 predictions for this patient cluster in a tight band,
while the deep tree's spray across the full span from 0 to 1 — and the average
per-patient standard deviation is `0.1460` for the shallow tree against `0.3061` for the
deep one, more than double.
**Infer:** the deep model's answer for a specific patient depends substantially on which
237 patients it happened to train on, which is variance in its most concrete form: the
model is reporting properties of its training sample, not of the patient in front of it.
For a clinical tool this is disqualifying independently of any accuracy metric — a
"risk" that would have been 0.2 had the registry been collected a month later is not a
risk estimate. Notice this failure is invisible to every diagnostic in Sessions 9-11,
all of which look at one fitted model; you have to refit to see it at all.

## Step 4 — Bias: how far off is the *average* prediction?

Averaging away the variance leaves what is systematically wrong: the gap between the
mean prediction and the true outcome.

In [ ]:
shallow_avg = shallow.mean(axis=0)
deep_avg = deep.mean(axis=0)

print(f"mean |average prediction - actual outcome|:")
print(f"  shallow: {np.abs(shallow_avg - y_test).mean():.4f}")
print(f"  deep:    {np.abs(deep_avg - y_test).mean():.4f}")

print(f"\ndistinct probabilities a SINGLE fit can emit:")
one_shallow = DecisionTreeClassifier(max_depth=1, random_state=0).fit(X_train, y_train)
one_deep = DecisionTreeClassifier(max_depth=None, random_state=0).fit(X_train, y_train)
print(f"  shallow: {np.unique(one_shallow.predict_proba(X_test)[:, 1]).round(3)}")
print(f"  deep:    {np.unique(one_deep.predict_proba(X_test)[:, 1]).round(3)}")
print("\ndistinct values after averaging 50 bootstrap fits:")
print(f"  shallow: {len(np.unique(shallow_avg.round(3)))}")
print(f"  deep:    {len(np.unique(deep_avg.round(3)))}")

**Observe:** the deep model's *averaged* predictions sit closer to the actual outcomes
(`0.241` vs `0.357`), and both single fits emit exactly two distinct probabilities — but
for opposite reasons: the shallow tree's are two intermediate rates (`0.214` and
`0.739`), the deep tree's are a flat `0.0` and `1.0`. After averaging 50 fits the deep
model spans 35 distinct values and the shallow one 16.
**Infer:** the shallow tree's two-value limit is bias made structural. A depth-1 tree
splits on one input and assigns every patient one of two probabilities, so any real
pattern involving a second input is unrepresentable no matter how many patients you
collect — that is what "does not shrink with more data" means, and it is why the cure
for bias is flexibility rather than volume. The deep tree's `0.0`/`1.0` output is the
opposite pathology: pure leaves mean it claims total certainty about every patient, and
which certainty it claims flips between resamples, which is variance. That averaging
alone turns those two values into a graded range is the mechanism Step 7 exploits. The tension is now explicit:
the same dial reduces one error source and inflates the other, which is the whole reason
Session 11's held-out curve had a peak instead of a slope.

## Step 5 — The decomposition across the complexity range

Expected squared error decomposes into $\text{bias}^2 + \text{variance} +
\text{irreducible noise}$. The first two are computable from the bootstrap matrices;
the third is not, and is what remains even for a perfect model.

In [ ]:
def decompose(predictions, y_true):
    average = predictions.mean(axis=0)
    bias_squared = np.mean((average - y_true) ** 2)
    variance = predictions.var(axis=0).mean()
    return bias_squared, variance

rows = []
for depth in [1, 2, 3, 5, 8, None]:
    preds = bootstrap_predictions(depth)
    b2, v = decompose(preds, y_test)
    rows.append({
        "max_depth": str(depth),
        "bias^2": b2,
        "variance": v,
        "bias^2 + var": b2 + v,
        "AUC of averaged prediction": roc_auc_score(y_test, preds.mean(axis=0)),
    })
decomposition = pd.DataFrame(rows)
print(decomposition.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
xs = range(len(decomposition))
ax.plot(xs, decomposition["bias^2"], "o-", color="steelblue", label="bias$^2$")
ax.plot(xs, decomposition["variance"], "o-", color="indianred", label="variance")
ax.plot(xs, decomposition["bias^2 + var"], "o-", color="black", lw=2, label="total")
ax.set_xticks(list(xs)); ax.set_xticklabels(decomposition["max_depth"])
ax.set_xlabel("max_depth (complexity increasing ->)"); ax.set_ylabel("error component")
ax.set_title("The tradeoff, measured on this registry")
ax.legend()
plt.tight_layout()
plt.show()

**Observe:** bias² falls from `0.157` at depth 1 to `0.121` by depth 3 and then
flattens, while variance rises steadily from `0.034` to `0.124` — nearly a fourfold
increase. Total error is lowest at depth 3 (`0.173`) and worst at unlimited depth
(`0.251`).
**Infer:** the two curves crossing is Session 11's peak, explained. Notice how quickly
bias exhausts its gains — nearly all of the reduction is bought by depth 2, and depths
beyond 3 purchase almost no bias reduction while continuing to pay full price in
variance. That asymmetry is why the optimum sits early, and it is a property of *this
registry's size*: with 3,000 patients each additional split would be estimated from
enough data that variance would grow far more slowly, and the optimum would move right.
Complexity is not a property of the problem alone; it is a property of the problem and
the amount of data together.

## Step 6 — Which one dominates here?

The decomposition is only useful if it changes what you do next. It says which of the
two cures to reach for.

In [ ]:
worst = decomposition.iloc[-1]
best = decomposition.loc[decomposition["bias^2 + var"].idxmin()]

print(f"at max_depth={worst['max_depth']}:")
print(f"  bias^2   = {worst['bias^2']:.4f}  ({100 * worst['bias^2'] / worst['bias^2 + var']:.0f}% of the total)")
print(f"  variance = {worst['variance']:.4f}  ({100 * worst['variance'] / worst['bias^2 + var']:.0f}% of the total)")
print(f"\nbest total error at max_depth={best['max_depth']}")
print()
print("diagnosis -> prescription")
print("  variance-dominated: simplify, regularise, get more data, or AVERAGE MANY MODELS")
print("  bias-dominated:     more flexibility, better inputs, a different model family")

**Observe:** at unlimited depth, variance accounts for roughly half the measured error
and bias² the other half — but variance is the component that quadrupled on the way
there, while bias² barely moved after depth 2.
**Infer:** the *marginal* picture is what decides the prescription, and it says
unambiguously that complexity here is buying variance and nothing else. So the fix is a
variance fix, and the interesting option is the last one on the list: averaging many
models. Session 2's variance-of-a-sum result is the reason it works — averaging $m$
predictions with pairwise correlation $\rho$ leaves variance proportional to
$\rho + (1-\rho)/m$, so the gain depends entirely on the models being *decorrelated*
from each other. Two corollaries follow immediately: averaging 50 copies of the same
deterministic tree achieves nothing, and averaging cannot reduce bias at all, since
every model is biased in the same direction. That is why this diagnosis has to come
first.

## Step 7 — Attacking variance directly: a random forest

A random forest is exactly the averaging argument implemented: fit many deep (low-bias,
high-variance) trees on bootstrap resamples, decorrelate them further by restricting
each split to a random subset of inputs, and average their predictions.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

single_tree = DecisionTreeClassifier(max_depth=None, random_state=0).fit(X_train, y_train)
forest = RandomForestClassifier(n_estimators=300, random_state=0).fit(X_train, y_train)
logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_train, y_train)

print(f"{'model':32} {'test AUC':>9}")
for name, m in [("single unrestricted tree", single_tree),
                ("random forest (300 trees)", forest),
                ("logistic regression", logistic)]:
    print(f"{name:32} {roc_auc_score(y_test, m.predict_proba(X_test)[:, 1]):>9.3f}")

forest_preds = np.array([t.predict_proba(X_test)[:, 1] for t in forest.estimators_])
print(f"\nvariance across the forest's individual trees: {forest_preds.var(axis=0).mean():.4f}")
print(f"variance of the forest's averaged prediction is what remains after averaging 300 of them")

**Observe:** the single unrestricted tree scores `0.732`, the forest of the same
unrestricted trees scores `0.942`, and logistic regression scores `0.950`.
**Infer:** averaging recovered `0.21` of AUC from components that are individually
terrible, without changing a single tree's depth — the diagnosis was variance and the
variance cure worked. The logistic regression edging out the forest is the more
interesting result: a thirteen-parameter linear model matching a 300-tree ensemble means
the disease boundary in this registry is close to linear, so the forest's extra
flexibility has nothing to find and it spends its capacity fitting noise the averaging
then has to remove. This is the practical form of "no free lunch" — model choice is a
claim about the shape of the problem, and on 297 patients with a near-linear boundary,
the simpler claim wins. It is also the better clinical tool for the reason Session 9
cared about: its coefficients can be read.

## Step 8 — Closing the loop

Twelve sessions, one pipeline. Each stage's failure mode, and where it would have
surfaced.

In [ ]:
pipeline = pd.DataFrame([
    ("1",  "Probability",        "misread P(disease|test) as P(test|disease)", "wrong risk at the bedside"),
    ("2",  "Random variables",   "summarise a skewed input with mean +/- sd",  "misleading input description"),
    ("3",  "Distributions",      "assume Normality without checking",          "invalid p-values in 6-8"),
    ("4",  "Sampling",           "convenience sample",                         "confident, biased, unfixable"),
    ("5",  "Correlation",        "trust r without plotting; miss redundancy",  "unstable coefficients in 9"),
    ("6",  "Hypothesis testing", "no correction for 13 tests",                 "a false predictor in the model"),
    ("7",  "t-test",             "classic t on unequal variances",             "wrong significance calls"),
    ("8",  "Chi-square",         "ignore small expected counts",               "an invalid p-value, silently"),
    ("9",  "Regression",         "report training R^2",                        "overstated performance"),
    ("10", "Train-test split",   "preprocess before splitting",                "AUC 0.80 from pure noise"),
    ("11", "Overfitting",        "pick complexity by training score",          "the worst model, shipped"),
    ("12", "Bias-variance",      "apply the wrong cure",                       "more complexity, more error"),
], columns=["#", "stage", "the mistake", "what it costs downstream"])
print(pipeline.to_string(index=False))

**Observe:** twelve stages, twelve failure modes, and a "what it costs" column in which
almost nothing is caught by the stage that caused it.
**Infer:** that displacement is the systems-thinking point the module was built around.
A Session 3 Normality assumption produces a wrong p-value in Session 7; a Session 4
sampling error produces a model that fails only in deployment; a Session 10 leak
produces an excellent-looking number that is pure fiction. None of these announce
themselves where they occur, which is why the discipline has to be applied at each stage
rather than caught by a final check — there is no final check that finds them. The
finished system: a cross-validated AUC near 0.89-0.95 with the fold spread attached, a
model whose coefficients can be explained, a stated prevalence assumption bounding where
it may be used, and an explicit list of what has *not* been established — calibration,
causality, and performance on any population but this one.

## Where this leads

- [4. ML Algorithms](../4.%20ML%20Algorithms) and
  [7. Machine Learning Fundamentals and Predictive Analytics](../7.%20Machine%20Learning%20Fundamentals%20and%20Predictive%20Analytics)
  build on the sampling, correlation, and hypothesis-testing material here.
- [11. MLOps Skilling Course](../11.%20MLOps%20Skilling%20Course) takes a validated model
  and addresses what this module explicitly could not: the "same era?" row of Session
  10's gap table becomes data-drift monitoring, and the reproducibility discipline
  becomes experiment tracking and data versioning.

## Try it yourself

1. Re-run Step 5 with `N_BOOTSTRAP = 200`. Do the bias and variance estimates stabilise,
   and which one was noisier at 50?
2. Run the whole decomposition on logistic regression with varying `C`. Is it
   bias-dominated or variance-dominated, and does that explain its Step 7 result?
3. Train the forest on 100 patients instead of 237. Does averaging recover as much, and
   what does that say about where variance comes from?
4. Compare the forest's predicted probabilities against observed disease rates in bins
   (a calibration curve). Session 10's last gap row said AUC ignores this — how well
   calibrated is the model that ranks best?